# Feature Engineering
In this notebook, we create new columns and target variables for training Machine Learning models:
1. `passenger_demand` -> in hourly_demand_operations.csv
2. `capacity_shortage` -> passenger_demand minus total_capacity (if negative, set to 0)
3. `next_hour_passenger_demand` -> shifted to next hour for same route and service class
4. `bunching_label` -> 1 if actual_headway_min < 0.60 * scheduled_headway_min, else 0

All final processed datasets are saved directly in `cleaned_data/`.


## 1. Load Data from `cleaned_data/`


In [1]:
import os
import pandas as pd
import numpy as np

# Auto-navigate to project root if running inside subfolder
if os.path.basename(os.getcwd()) in ["data preprocessing", "EDA"]:
    os.chdir("..")

df_demand = pd.read_csv("cleaned_data/hourly_demand_operations.csv", low_memory=False)
df_trips = pd.read_csv("cleaned_data/trip_operations.csv", low_memory=False)

print("Demand shape:", df_demand.shape)
print("Trips shape:", df_trips.shape)


Demand shape: (74470, 33)
Trips shape: (319375, 12)


## 2. Feature Engineering for Hourly Passenger Demand


In [2]:
# 1. Passenger demand column
df_demand["passenger_demand"] = df_demand["observed_passenger_count_raw"]

# 2. Capacity shortage column (if negative, set to zero)
df_demand["capacity_shortage"] = (df_demand["passenger_demand"] - df_demand["total_capacity"]).clip(lower=0)

# 3. Sort values by route, service class, date, hour
df_demand = df_demand.sort_values(["route_id", "service_class", "date", "hour"])

# 4. Target: next hour passenger demand (shifted by -1)
df_demand["next_hour_passenger_demand"] = (
    df_demand.groupby(["route_id", "service_class"])["observed_passenger_count_raw"]
      .shift(-1)
)

# Extract hour integer for modeling
df_demand["hour_int"] = df_demand["hour"].apply(lambda x: int(str(x).split(":")[0]) if ":" in str(x) else int(x))

print("Demand features engineered:")
print(df_demand[["route_id", "service_class", "date", "hour", "passenger_demand", "total_capacity", "capacity_shortage", "next_hour_passenger_demand"]].head())
print("\nMissing values in next_hour_passenger_demand:", df_demand["next_hour_passenger_demand"].isnull().sum())
print("(Note: These boundary rows are excluded during model training, not filled with zero or mean)")


Demand features engineered:
  route_id service_class        date   hour  passenger_demand  total_capacity  \
0      I01     Intercity  2026-01-01  06:00                15              80   
1      I01     Intercity  2026-01-01  07:00                20              80   
2      I01     Intercity  2026-01-01  08:00                30              80   
3      I01     Intercity  2026-01-01  09:00                32              80   
4      I01     Intercity  2026-01-01  10:00                40              80   

   capacity_shortage  next_hour_passenger_demand  
0                  0                        20.0  
1                  0                        30.0  
2                  0                        32.0  
3                  0                        40.0  
4                  0                        35.0  

Missing values in next_hour_passenger_demand: 12
(Note: These boundary rows are excluded during model training, not filled with zero or mean)


## 3. Feature Engineering for Trip Operations (Bus Bunching)


In [3]:
# Target: bunching_label based on 60% headway threshold rule
# If actual headway < 60% of scheduled headway, trip is bunching (1), else (0)
df_trips["bunching_label"] = (df_trips["actual_headway_min"] < 0.60 * df_trips["scheduled_headway_min"]).astype(int)

# Headway deviation and ratio
df_trips["headway_deviation_min"] = df_trips["actual_headway_min"] - df_trips["scheduled_headway_min"]
df_trips["headway_ratio"] = df_trips["actual_headway_min"] / df_trips["scheduled_headway_min"]
df_trips["hour_int"] = df_trips["hour"].apply(lambda x: int(str(x).split(":")[0]) if ":" in str(x) else int(x))

print("Bunching Label Distribution:")
print(df_trips["bunching_label"].value_counts())
print(df_trips["bunching_label"].value_counts(normalize=True) * 100)


Bunching Label Distribution:


bunching_label
0    310482
1      8893
Name: count, dtype: int64
bunching_label
0    97.215499
1     2.784501
Name: proportion, dtype: float64


## 4. Save Final Engineered Datasets to `cleaned_data/`


In [4]:
# Save directly to cleaned_data/
df_demand.to_csv("cleaned_data/hourly_demand_operations.csv", index=False)
df_trips.to_csv("cleaned_data/trip_operations.csv", index=False)

print("Final engineered datasets saved directly in 'cleaned_data/' folder!")


Final engineered datasets saved directly in 'cleaned_data/' folder!
